In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import HTML, Markdown, display
import pandas as pd
import plotly.io as pio

here = Path.cwd().resolve()
ancestors = (here, *here.parents)
override = os.environ.get('POSTER_EVIDENCE')
repo = next((p for p in ancestors if (p / 'src/evaluation/report.py').is_file()), None)
if repo is None:
    raise FileNotFoundError('Start Jupyter from the repository or its notebooks directory.')
sys.path.insert(0, str(repo / 'src'))
from evaluation.report import (
    COMPARISON_NOTE, CONTROLS_NOTE, FILTER_NOTE, GRAPH_CONFIG, METRIC_NOTE, MODELS, OUTSEQ_NOTE,
    PROTOCOL, QUESTION, RESULT_NOTE, SIGNAL_NOTE, SLIDE_NOTE, SPLIT_NOTE, TASK_NOTE,
    figures, headline_table, interpretation_notes, load_evidence, outseq_table,
    signal_table, slide_table, winner_table,
)

evidence = load_evidence(Path(override) if override else repo / 'poster/evidence')
display(evidence['manifest'])
plots = figures(evidence)
pio.renderers.default = 'plotly_mimetype+notebook'  # Inline Plotly figures, including offline output.

def show(name):
    plots[name].show(config=GRAPH_CONFIG)

# Direct versus learned hippocampal LFP features

## 1 · Question and workflow

In [ ]:
display(Markdown('\n\n'.join((QUESTION, COMPARISON_NOTE, RESULT_NOTE, TASK_NOTE))))
show('workflow')

## 2 · Matched modeling protocol and 60 Hz quality checks

In [ ]:
display(HTML(pd.DataFrame(PROTOCOL, columns=['Choice', *MODELS.values()]).to_html(index=False)))
display(Markdown('\n\n'.join((CONTROLS_NOTE, SPLIT_NOTE, FILTER_NOTE))))
display(pd.DataFrame(signal_table(evidence)))
display(Markdown(SIGNAL_NOTE))

## 3 · Reviewed results and historical cross-check

The primary paired comparison uses the common 500 ms windows; the per-rat lines connect each model on matched trial labels. Repeated model predictions are not independent replications, and we do not infer a population interval from these five previously viewed rats.

In [ ]:
display(pd.DataFrame(headline_table(evidence)))
display(Markdown(METRIC_NOTE))
show('paired')

### Archived finalist scores · different configurations

In [ ]:
display(pd.DataFrame(slide_table(evidence)))
display(Markdown(SLIDE_NOTE))

## 4 · Interpretability: training and validation only

**Signed L1 coefficients.** The panel uses class 3 of Mitt's position model. Each tile is the arithmetic mean of centered, standardized signed coefficients over all electrodes, including zeros; opposite signs can cancel. Hover to see the number of nonzero electrodes.

**Grouped masking.** For each rat and target, set one group to zero in head-standardized units (its TRAIN feature mean). Changes are intact minus masked validation BA using the same fitted estimator and labels. The three registered frequency groups and two time groups were fixed; this is sensitivity, not refitting, feature selection or a test ablation.

In [ ]:
show('weights')
display(Markdown(interpretation_notes(evidence)['weights']))
show('masking')

### CEBRA centroids · Mitt position model

One PCA fitted on TRAIN projects validation once. Both panels use the same coordinates and axes, grouped by odor or position; marker area reflects support, and empty classes are omitted. Switch between all validation trials and the OutSeq subset without changing the axes or projection. PCA is visualization only; the frozen classifier uses all eight representation dimensions. On InSeq trials odor and position co-vary, so these plots do not establish biological specificity.

In [ ]:
show('projection')
display(Markdown(interpretation_notes(evidence)['projection']))

### OutSeq diagnostic · retrospective, separate from interpretability

In [ ]:
show('outseq')
display(pd.DataFrame(outseq_table(evidence)))
display(Markdown(OUTSEQ_NOTE))

## 5 · Separate model-matrix benchmark and limitations

**Grouped input masking** measures validation sensitivity for a fixed amplitude estimator. The **encoder–decoder matrix** was a separate validation search: 15 encoders × 13 decoders per target, 66 scheduled per target, and two historical anchors. This release renders its reviewed aggregates; it does not rerun the search. Gray cells have no validation score. Hover for status and exact input view. The rankings are descriptive, not independent confirmation.

In [ ]:
counts = evidence['manifest']['counts']
display(Markdown(f"**Audit:** {counts['matrix_slots']} total rows; {counts['scheduled_pairs']} scheduled pairs; "
                 f"{counts['historical_anchors']} fixed historical anchors; "
                 f"{counts['terminal_tuning_trials']} terminal tuning trials."))
display(pd.DataFrame(winner_table(evidence)))
show('matrix')

### Reproduce: aggregate rebuild or private frozen replay

This notebook and the standalone report use the same checksummed aggregate packet; neither needs recordings, weights, or a GPU. See the [repository README](https://github.com/zslrmhb/hippocampal-representation-learning#1-poster-rebuild-from-reviewed-aggregates) for the locked HTML, PDF, and notebook build commands.

Private scientific replay is separate. With authorized recordings and frozen checkpoints, it checks validation parity before any retrospective test scoring; `--interpretability-only` stops before scoring test. The checkout code revision updates implementation hashes without changing the original evidence packet, cohorts, or expected results.